# Essential Commands for the Capstone — Python Demo

Dataset: **Campus Wellness Survey** (synthetic)

## Load the data

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

df = pd.read_csv("campus_wellness.csv")
df.head()

## Explore structure

In [ ]:
df.shape

In [ ]:
df.dtypes

In [ ]:
df.info()

## Data wrangling basics

### Selecting columns

In [ ]:
# Single column (returns a Series)
df["sleep_hrs"]

In [ ]:
# Multiple columns (returns a DataFrame)
df[["sleep_hrs", "coffee_cups", "stress_score"]]

### Selecting / filtering rows

In [ ]:
# By position — first 5 rows
df.iloc[:5]

In [ ]:
# By condition — only students who do yoga
df[df["yoga"] == 1]

In [ ]:
# Combine conditions: yoga practitioners under 22
df[(df["yoga"] == 1) & (df["age"] < 22)]

### Column-level aggregates

In [ ]:
# Sum, mean, median of one column
print("Sum:   ", df["sleep_hrs"].sum())
print("Mean:  ", df["sleep_hrs"].mean())
print("Median:", df["sleep_hrs"].median())

In [ ]:
# Across several columns at once
df[["sleep_hrs", "coffee_cups", "exercise_hrs"]].agg(["sum", "mean", "median"])

## Missing values

In [ ]:
# Missing count per column
df.isnull().sum()

In [ ]:
# Rows with at least one missing value
df.isnull().any(axis=1).sum()

## Five-point summary (quantitative variables)

In [ ]:
quant = ["sleep_hrs", "coffee_cups", "exercise_hrs", "stress_score", "age", "study_hrs"]
df[quant].describe()

## Counts & proportions (categorical variables)

In [ ]:
# Counts
df["sex"].value_counts().sort_index()

In [ ]:
df["yoga"].value_counts().sort_index()

In [ ]:
df["exam_pass"].value_counts().sort_index()

In [ ]:
df["semester"].value_counts().sort_index()

In [ ]:
# Proportions
df["sex"].value_counts(normalize=True).sort_index()

In [ ]:
df["yoga"].value_counts(normalize=True).sort_index()

In [ ]:
df["exam_pass"].value_counts(normalize=True).sort_index()

## Univariate visualizations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, var in zip(axes, ["sleep_hrs", "stress_score", "exercise_hrs"]):
    df[var].hist(bins=25, edgecolor="black", ax=ax)
    ax.set_title(var)
plt.tight_layout()
plt.show()

## Bar plot (categorical variable)

In [ ]:
df["semester"].value_counts().sort_index().plot(kind="bar", edgecolor="black")
plt.xlabel("Semester"); plt.ylabel("Count")
plt.title("Students by Semester")
plt.tight_layout(); plt.show()

## Table 1 — stratified by sex

In [ ]:
# pip install tableone
from tableone import TableOne

tab1 = TableOne(
    df,
    columns=["sleep_hrs","coffee_cups","exercise_hrs","stress_score",
             "yoga","exam_pass","age","study_hrs"],
    categorical=["yoga","exam_pass"],
    groupby="sex",
    pval=True
)
print(tab1.tabulate(tablefmt="grid"))

## Welch's t-test — stress_score by sex

In [ ]:
male = df.loc[df["sex"]==1, "stress_score"].dropna()
female = df.loc[df["sex"]==2, "stress_score"].dropna()
stats.ttest_ind(male, female, equal_var=False)

In [ ]:
df.boxplot(column="stress_score", by="sex")
plt.suptitle("")
plt.title("Stress Score by Sex")
plt.show()

## Correlation — stress_score vs sleep_hrs

In [ ]:
stats.pearsonr(df["stress_score"].dropna(), df["sleep_hrs"])

In [ ]:
plt.scatter(df["sleep_hrs"], df["stress_score"], alpha=0.3)
plt.xlabel("Sleep Hours"); plt.ylabel("Stress Score")
plt.title("Stress Score vs Sleep Hours")
plt.show()

## Multiple linear regression — stress_score ~ sex + sleep_hrs + exercise_hrs + yoga

In [ ]:
import statsmodels.api as sm

sub = df[["sex","sleep_hrs","exercise_hrs","yoga","stress_score"]].dropna()
X = sm.add_constant(sub[["sex","sleep_hrs","exercise_hrs","yoga"]])
y = sub["stress_score"]
model = sm.OLS(y, X).fit()
print(model.summary())

## Logistic regression — exam_pass ~ sex + sleep_hrs + yoga + study_hrs

In [ ]:
sub = df[["sex","sleep_hrs","yoga","study_hrs","exam_pass"]].dropna()
X = sm.add_constant(sub[["sex","sleep_hrs","yoga","study_hrs"]])
y = sub["exam_pass"]
logit = sm.Logit(y, X).fit()
print(logit.summary())

In [ ]:
# Odds ratios + confidence intervals
print("\nOdds Ratios:")
print(np.exp(logit.params))
print("\n95% CI:")
print(np.exp(logit.conf_int()))

## Stratified correlation — exercise_hrs vs age, by sex

In [ ]:
for s, label in [(1, "Male"), (2, "Female")]:
    sub = df[df["sex"]==s].dropna(subset=["exercise_hrs","age"])
    r, p = stats.pearsonr(sub["exercise_hrs"], sub["age"])
    print(f"{label}: r = {r:.3f}, p = {p:.4f}")